In [ ]:
import os
import soundfile as sf
from datasets import load_from_disk
import torch
import torchaudio
from models import voicecraft
import numpy as np
import random
import json
import requests
from encodec import EncodecModel
from encodec.utils import convert_audio
from hebrew import Hebrew
from hebrew.chars import HebrewChar
import pickle

In [3]:
# Check the sampling rate and format

# Load directly from  local folder
dataset = load_from_disk("./fleurs_hebrew")
sample_audio = dataset[0]['audio']

# VoiceCraft usually needs 16000Hz. Let's confirm.
print(f"Current Sampling Rate: {sample_audio['sampling_rate']} Hz")

Current Sampling Rate: 16000 Hz


In [ ]:
# Export few samples to WAV files for VoiceCraft

# 1. Load your saved dataset
# Make sure this path matches the folder where you did 'save_to_disk'
ds_path = "./fleurs_hebrew" 
ds = load_from_disk(ds_path)

# 2. Create a folder for the WAV files
output_dir = "./voicecraft_samples"
os.makedirs(output_dir, exist_ok=True)

# 3. Export the first 5 samples
print(f"Exporting samples to {output_dir}...")

for i in range(5):
    sample =  [i]
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    text = sample['transcription']
    
    file_path = os.path.join(output_dir, f"sample_{i}.wav")
    
    # Save as WAV
    sf.write(file_path, audio_array, sr)
    
    print(f"--- Sample {i} ---")
    print(f"File: {file_path}")
    print(f"SR: {sr}Hz | Text: {text}")

# VoiceCraft check:
if sr == 16000:
    print("\n✅ Success! Sampling rate is 16kHz, perfect for VoiceCraft.")
else:
    print(f"\n⚠️ Note: SR is {sr}Hz. VoiceCraft might need 16kHz.")

Exporting samples to ./voicecraft_samples...
--- Sample 0 ---
File: ./voicecraft_samples/sample_0.wav
SR: 16000Hz | Text: זו הרכישה הגדולה ביותר בתולדות ebay
--- Sample 1 ---
File: ./voicecraft_samples/sample_1.wav
SR: 16000Hz | Text: כמו כן מטייל בריטי בספרד עלול להבין בטעות נפנוף לשלום בכף יד הפונה לכיוון המנופף ולא לכיוון האדם שאליו מנופפים כמחווה לחזור בחזרה
--- Sample 2 ---
File: ./voicecraft_samples/sample_2.wav
SR: 16000Hz | Text: בסוף 2015 הקימה טוגי-נט את אסטרו-נט רדיו כתחנת בת
--- Sample 3 ---
File: ./voicecraft_samples/sample_3.wav
SR: 16000Hz | Text: בקשר למצב העולמי הפיננסי זפאטרו המשיך ואמר ש המערכת הפיננסית היא חלק מהכלכלה חלק חיוני שלה
--- Sample 4 ---
File: ./voicecraft_samples/sample_4.wav
SR: 16000Hz | Text: החגיגות התחילו במופע מיוחד של סירק דה סוליי להקה המפורסמת בעולם כולו

✅ Success! Sampling rate is 16kHz, perfect for VoiceCraft.


In [4]:
# Test if the tokenizer handles Hebrew
from data.tokenizer import AudioTokenizer, TextTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model, tokenizer, and weights
voicecraft_name = "giga330M.pth"
ckpt_fn = f"./pretrained_models/{voicecraft_name}"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

ckpt = torch.load(ckpt_fn, map_location="cpu")
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
phn2num = ckpt['phn2num']

# phn2num = ckpt['phn2num']
text_tokenizer = TextTokenizer(backend="espeak")
# Re-initialize the tokenizer with Hebrew support
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")
audio_tokenizer = AudioTokenizer(signature=encodec_fn) 

# English comments: Test if tokenizer can process Hebrew text
test_text = "שלום, מה נשמע?"
try:
    tokens = text_tokenizer_he(test_text)
    print(f"Text: {test_text}")
    print(f"Tokens/Phonemes: {tokens}")
except Exception as e:
    print(f"Error: {e}")

Dora directory: /tmp/audiocraft_sukiennik
/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Text: שלום, מה נשמע?
Tokens/Phonemes: [['ʃ', 'a', 'l', 'o', 'm', ',', '_', 'm', 'a', '_', 'n', 'ʃ', 'm', 'ʔ', '?']]


In [5]:
# List all available methods for the tokenizer object
print(dir(text_tokenizer))

['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'backend', 'separator', 'to_list']


In [6]:
# Test a complex sentence from FLEURS
sample_idx = 0 # The eBay sentence
original_text = dataset[sample_idx]['transcription']
phonetic_result = text_tokenizer_he(original_text)

print(f"Original: {original_text}")
print(f"Phonetic: {phonetic_result}")

Original: זו הרכישה הגדולה ביותר בתולדות ebay
Phonetic: [['z', 'v', '_', 'ʔ', 'ʁ', 'χ', 'i', 'ʃ', 'ʔ', '_', 'ʔ', 'ɡ', 'd', 'v', 'l', 'ʔ', '_', 'v', 'i', 'v', 't', 'ʁ', '_', 'v', 't', 'v', 'l', 'd', 'v', 't', '_', '(', 'en', ')', 'iː', '_', 'b', 'iː', '_', 'eɪ', '_', 'w', 'aɪ', '(', 'he', ')']]


In [7]:
# Test how Niqqud changes phonemes for the word 'Sefer/Sapar/Siper'
test_cases = {
    "ספר (No Niqqud)": "ספר",
    "סֵפֶר (Book - Sefer)": "סֵפֶר",
    "סַפָּר (Barber - Sapar)": "סַפָּר",
    "סִפֵּר (Told - Siper)": "סִפֵּר"
}

print("--- Testing Phonemes with Niqqud ---")
for description, text in test_cases.items():
    phonemes = text_tokenizer_he(text)
    print(f"{description} -> {phonemes}")

--- Testing Phonemes with Niqqud ---
ספר (No Niqqud) -> [['s', 'e', 'f', 'e', 'ʁ']]
סֵפֶר (Book - Sefer) -> [['s', 'e', 'f', 'e', 'ʁ']]
סַפָּר (Barber - Sapar) -> [['s', 'a', 'm', 'e', 'χ', '_', 'a', '_', 'f', 'e', '_', 'a', '_', '(', 'en', ')', 'h', 'iː', 'b', 'ɹ', 'uː', 'd', 'a', 'ɡ', 'ɛ', 'ʃ', '(', 'he', ')', '_', 'ʁ', 'e', 'ʃ']]
סִפֵּר (Told - Siper) -> [['s', 'a', 'm', 'e', 'χ', '_', 'i', '_', 'f', 'e', '_', 'e', '_', '(', 'en', ')', 'h', 'iː', 'b', 'ɹ', 'uː', 'd', 'a', 'ɡ', 'ɛ', 'ʃ', '(', 'he', ')', '_', 'ʁ', 'e', 'ʃ']]


In [8]:
# Testing only vowels, NO dagesh (the dot inside letters)
test_clean = {
    "Sapar (No Dagesh)": "סַפָר", 
    "Siper (No Dagesh)": "סִפֵר"
}

for desc, text in test_clean.items():
    print(f"{desc} -> {text_tokenizer_he(text)}")

Sapar (No Dagesh) -> [['s', 'a', 'f', 'a', 'ʁ']]
Siper (No Dagesh) -> [['s', 'i', 'f', 'e', 'ʁ']]


In [9]:
# hyperparameters for inference
codec_audio_sr = 16000
codec_sr = 50
top_k = 0
top_p = 0.8
temperature = 1
silence_tokens=[1388,1898,131]
kvcache = 1 # NOTE if OOM, change this to 0, or try the 330M model

# NOTE adjust the below three arguments if the generation is not as good
stop_repetition = 3 # NOTE if the model generate long silence, reduce the stop_repetition to 3, 2 or even 1
sample_batch_size = 3 # for gigaHalfLibri330M_TTSEnhanced_max16s.pth, 1 or 2 should be fine since the model is trained to do TTS, for the other two models, might need a higher number. NOTE: if the if there are long silence or unnaturally strecthed words, increase sample_batch_size to 5 or higher. What this will do to the model is that the model will run sample_batch_size examples of the same audio, and pick the one that's the shortest. So if the speech rate of the generated is too fast change it to a smaller number.
seed = 1 # change seed if you are still unhappy with the result

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything(seed)

decode_config = {
    'top_k': top_k,
    'top_p': top_p,
    'temperature': temperature,
    'stop_repetition': stop_repetition,
    'kvcache': kvcache, 
    "codec_audio_sr": codec_audio_sr,
    "codec_sr": codec_sr, 
    "silence_tokens": silence_tokens, 
    "sample_batch_size": sample_batch_size}

In [10]:
# Prepare the inputs for the first Hebrew generation
# 1. Path to one of the 16kHz files we created
audio_fn = "fleurs_hebrew/voicecraft_samples/sample_0.wav"

# 2. The transcript from FLEURS (without Niqqud for now, to see if it works)
original_prompt_transcript = "זו הרכישה הגדולה ביותר בתולדות ebay"

# 3. The text you want the model to say in that voice
target_transcript = "אני חושבת שתהיה תקיפה בחמישי בלילה"
target_transcript_test = "ani khoshevet she-ti-ye tkifa be-khamishi ba-layla"

# 4. Run the model (this is a general example, adjust to your specific function name)
# Usually it looks something like this:
# output_audio = model.generate(
#     text=target_text,
#     ref_audio=test_audio_path,
#     transcript=transcript,
#     tokenizer=text_tokenizer_he
# )



# 2. הטקסט החדש שאת רוצה שהדמות תגיד

# 3. חישוב ה-prompt_end_frame
# VoiceCraft צריך לדעת איפה נגמר השמע של הרפרנס. 
# בדרך כלל משתמשים בכל אורך הקובץ אם רוצים את כל הקול כרפרנס.
info = torchaudio.info(audio_fn)
prompt_end_frame = info.num_frames 

# 4. הגדרות קידוד (נשתמש בברירת המחדל אם כבר הגדרת אותן קודם ב-Notebook)
# אם decode_config לא מוגדר, בדרך כלל הוא נראה כך:
# decode_config = {'top_k': 0, 'top_p': 0.8, 'temperature': 1, 'stop_token': phn2num['<end_of_text>']}

In [11]:
# Run the actual inference
from inference_tts_scale import inference_one_sample

with torch.no_grad():
    with torch.cuda.amp.autocast():
        concated_audio, gen_audio = inference_one_sample(
            model, 
            ckpt["config"], 
            phn2num, 
            # text_tokenizer,  # משתמשים ב-Tokenizer העברי שלנו!
            text_tokenizer_he,  # משתמשים ב-Tokenizer העברי שלנו!
            audio_tokenizer, 
            audio_fn, 
            #target_transcript, 
            target_transcript_test,
            device, 
            decode_config, 
            prompt_end_frame
        )

concated_audio, gen_audio = concated_audio[0].cpu(), gen_audio[0].cpu()

# display the audio
from IPython.display import Audio
print("concatenate prompt and generated:")
display(Audio(concated_audio, rate=codec_audio_sr))

print("generated:")
display(Audio(gen_audio, rate=codec_audio_sr))

/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


concatenate prompt and generated:


generated:


In [12]:
# Check if phonemes exist in the model's vocabulary
tokens = text_tokenizer_he(target_transcript)
print(f"Phonemes: {tokens}")

for p in tokens[0]:
    if p not in phn2num:
        print(f"⚠️ Phoneme '{p}' is MISSING from phn2num!")

Phonemes: [['a', 'n', 'i', '_', 'χ', 'v', 'ʃ', 'v', 't', '_', 'ʃ', 't', 'i', 'ʔ', '_', 't', 'k', 'i', 'f', 'ʔ', '_', 'v', 'χ', 'm', 'i', 'ʃ', 'j', '_', 'v', 'l', 'i', 'l', 'ʔ']]
⚠️ Phoneme 'a' is MISSING from phn2num!
⚠️ Phoneme 'χ' is MISSING from phn2num!
⚠️ Phoneme 'χ' is MISSING from phn2num!


In [12]:
# run again 27/02/2026 15:51
# Check if phonemes exist in the model's vocabulary
tokens = text_tokenizer_he(target_transcript)
print(f"Phonemes: {tokens}")

for p in tokens[0]:
    if p not in phn2num:
        print(f"⚠️ Phoneme '{p}' is MISSING from phn2num!")

Phonemes: [['a', 'n', 'i', '_', 'χ', 'v', 'ʃ', 'v', 't', '_', 'ʃ', 't', 'i', 'ʔ', '_', 't', 'k', 'i', 'f', 'ʔ', '_', 'v', 'χ', 'm', 'i', 'ʃ', 'j', '_', 'v', 'l', 'i', 'l', 'ʔ']]
⚠️ Phoneme 'a' is MISSING from phn2num!
⚠️ Phoneme 'χ' is MISSING from phn2num!
⚠️ Phoneme 'χ' is MISSING from phn2num!


In [ ]:
# can delete ?
!pip install nakdimon==0.1.1

ERROR: Ignored the following versions that require a different python version: 0.1.0 Requires-Python >=3.11; 0.1.1 Requires-Python >=3.11; 0.1.2 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement nakdimon==0.1.1 (from versions: none)
ERROR: No matching distribution found for nakdimon==0.1.1


In [ ]:
# can delete ?
!pip install hebrew

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grapheme: filename=grapheme-0.6.0-py3-none-any.whl size=210137 sha256=9b3fd8769b815efbe0d80274d40065dca308b6a46187891eb9a929ec24d0c4b0
  Stored in directory: /home/sukiennik/.cache/pip/wheels/01/e1/49/37e6bde9886439057450c494a79b0bef8bbe897a54aebfc757
Successfully built grapheme
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [hebrew]


In [ ]:
# can delete ?

# Function to get Niqqud via Dicta API (no installation needed)

def get_niqqud_via_api(text):
    # English comments: Sending text to Dicta API for automatic vocalization
    url = 'https://nakdan-3-0.loadbalancer.dicta.org.il/api/nakdan/process'
    payload = {
        "text": text,
        "genre": "modern",  # Can be 'modern', 'rabbinic', or 'poetry'
        "data": []
    }
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            # Extracting the vocalized text from Dicta's response format
            result = response.json()
            full_text = "".join([res['options'][0]['w'] + res['sep'] for res in result])
            return full_text
        return text
    except:
        return text

def clean_dagesh_v2(text):
    # English comments: Remove Dagesh (\u05bc) as it confuses the tokenizer
    return text.replace('\u05bc', '')

# --- Main Manifest Creation Loop ---
num_samples = 100
manifest = []

print(f"Starting to prepare manifest for {num_samples} samples...")

for i in range(num_samples):
    try:
        sample = dataset[i]
        raw_text = sample['transcription']
        
        # 1. Get Niqqud from API
        vocalized_text = get_niqqud_via_api(raw_text)
        
        # 2. Clean Dagesh
        clean_text = clean_dagesh_v2(vocalized_text)
        
        # 3. Get Phonemes
        phonemes_list = text_tokenizer_he(clean_text)[0]
        phonemes_str = " ".join(phonemes_list)
        
        manifest.append({
            "audio_filepath": f"voicecraft_samples/sample_{i}.wav",
            "text": clean_text,
            "phonemes": phonemes_str
        })
        
        if i % 10 == 0: print(f"Processed {i} samples...")
            
    except Exception as e:
        print(f"Error in sample {i}: {e}")

# Save the manifest
with open("hebrew_train_manifest.jsonl", "w", encoding="utf-8") as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Done! Check 'hebrew_train_manifest.jsonl'")

Starting to prepare manifest for 100 samples...


Processed 0 samples...


Processed 10 samples...


Processed 20 samples...


Processed 30 samples...


...

Processed 60 samples...


Processed 70 samples...


Processed 80 samples...


Processed 90 samples...


Done! Check 'hebrew_train_manifest.jsonl'


In [ ]:
# didn't run - check if necesary 
# English comments: Script using 'hebrew' library to clean text for manifest
from hebrew import Hebrew
from hebrew.chars import HebrewChar

def clean_hebrew_text(raw_text):
    # English comments: Standardize Hebrew text and remove non-Hebrew artifacts
    h = Hebrew(raw_text)
    # This removes extra spaces and ensures text is in a standard format
    clean = h.normalize().string
    return clean

def prepare_manifest_v2(dataset, output_file, num_samples=100):
    manifest = []
    print(f"Processing {num_samples} samples with 'hebrew' library...")
    
    for i in range(num_samples):
        try:
            sample = dataset[i]
            text = clean_hebrew_text(sample['transcription'])
            audio_path = f"voicecraft_samples/sample_{i}.wav"
            
            # Get phonemes (espeak-ng will guess the vowels)
            phonemes_list = text_tokenizer_he(text)[0]
            phonemes_str = " ".join(phonemes_list)
            
            manifest.append({
                "audio_filepath": audio_path,
                "text": text,
                "phonemes": phonemes_str
            })
        except Exception as e:
            print(f"Error in sample {i}: {e}")

    with open(output_file, 'w', encoding='utf-8') as f:
        for item in manifest:
            import json
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print("Manifest created successfully!")

# Run for the first 100 samples
prepare_manifest_v2(dataset, "hebrew_train_manifest.jsonl", num_samples=100)

In [13]:
# Inspect the first line of the created manifest
with open("hebrew_train_manifest.jsonl", "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())
    print("Text:", first_line['text'])
    print("Phonemes:", first_line['phonemes'])

Text: זו הרכישה הגדולה ביותר בתולדות ebay
Phonemes: z v _ ʔ ʁ χ i ʃ ʔ _ ʔ ɡ d v l ʔ _ v i v t ʁ _ v t v l d v t _ ( en ) iː _ b iː _ eɪ _ w aɪ ( he )


In [ ]:
# can delete ?
# check after doing pre-niqud

# Check how the API vocalized the first sample
with open("hebrew_train_manifest.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
    print(f"Original Text with Niqqud: {sample['text']}")
    print(f"Resulting Phonemes: {sample['phonemes']}")

Original Text with Niqqud: זו הרכישה הגדולה ביותר בתולדות ebay
Resulting Phonemes: z v _ ʔ ʁ χ i ʃ ʔ _ ʔ ɡ d v l ʔ _ v i v t ʁ _ v t v l d v t _ ( en ) iː _ b iː _ eɪ _ w aɪ ( he )


In [16]:
# English comments: Test how the phonemizer handles a single vocalized word
test_word = "זוֹ" # Zo with Holam
test_phonemes = text_tokenizer_he(test_word)[0]
print(f"Word: {test_word} -> Phonemes: {test_phonemes}")

test_word_2 = "זֶה" # Ze with Segol
test_phonemes_2 = text_tokenizer_he(test_word_2)[0]
print(f"Word: {test_word_2} -> Phonemes: {test_phonemes_2}")

Word: זוֹ -> Phonemes: ['z', 'o']
Word: זֶה -> Phonemes: ['z', 'e', 'ʔ']


In [ ]:
# check if can be delete ?
# Fixed manifest creation script to ensure Niqqud is applied

def prepare_manifest_final(dataset, output_file, num_samples=100):
    manifest = []
    print(f"Processing {num_samples} samples with Niqqud API...")
    
    for i in range(num_samples):
        try:
            raw_text = dataset[i]['transcription']
            
            # 1. Get Niqqud (The most important step!)
            vocalized = get_niqqud_via_api(raw_text)
            
            # 2. Clean Dagesh (Using the function we defined earlier)
            clean_text = clean_dagesh_v2(vocalized)
            
            # 3. Phonemize (Now it will see 'z o' instead of 'z v')
            phonemes_list = text_tokenizer_he(clean_text)[0]
            phonemes_str = " ".join(phonemes_list)
            
            # Optional: Print the first one to verify
            if i == 0:
                print(f"Verification - Text: {clean_text}")
                print(f"Verification - Phns: {phonemes_str}")
            
            manifest.append({
                "audio_filepath": f"voicecraft_samples/sample_{i}.wav",
                "text": clean_text,
                "phonemes": phonemes_str
            })
            
            if i % 10 == 0:
                print(f"Progress: {i}/{num_samples}")
                
        except Exception as e:
            print(f"Error in sample {i}: {e}")

    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in manifest:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    
    print(f"--- Manifest ready: {output_file} ---")

# Run this now
prepare_manifest_final(dataset, "hebrew_train_manifest.jsonl", num_samples=100)

Processing 100 samples with Niqqud API...


Verification - Text: זו הרכישה הגדולה ביותר בתולדות ebay
Verification - Phns: z v _ ʔ ʁ χ i ʃ ʔ _ ʔ ɡ d v l ʔ _ v i v t ʁ _ v t v l d v t _ ( en ) iː _ b iː _ eɪ _ w aɪ ( he )
Progress: 0/100


Progress: 10/100


Progress: 20/100


Progress: 30/100


In [ ]:
# Testing the updated Dicta API endpoint

def probe_dicta_api():
    url = "https://nakdan-3-0.loadbalancer.dicta.org.il/api/nakdan/process"
    payload = {
        "text": "שלום",
        "genre": "modern",
        "data": []
    }
    try:
        response = requests.post(url, json=payload, timeout=10)
        if response.status_code == 200:
            print("Successfully connected to Dicta API!")
            return True
        else:
            print(f"Server reached but returned status: {response.status_code}")
            return False
    except Exception as e:
        print(f"Could not reach server: {e}")
        return False

probe_dicta_api()

Server reached but returned status: 404


False

In [ ]:
# Final attempt at vocalization optimization using 'hebrew' library
from hebrew import Hebrew
from hebrew.chars import HebrewChar

def finalize_text_for_phonemizer(text):
    # English comments: Normalize Hebrew text to help the phonemizer handle it better
    h = Hebrew(text)
    # Normalize will handle issues like double-vav and other Unicode inconsistencies
    normalized = h.normalize().string
    return normalized

def prepare_manifest_final_attempt(dataset, output_file, num_samples=100):
    manifest = []
    print(f"Running final manifest prep for {num_samples} samples...")
    
    for i in range(num_samples):
        try:
            raw_text = dataset[i]['transcription']
            
            # 1. Normalize text
            clean_text = finalize_text_for_phonemizer(raw_text)
            
            # 2. Phonemize
            phonemes_list = text_tokenizer_he(clean_text)[0]
            phonemes_str = " ".join(phonemes_list)
            
            # 3. Quick check: does 'זו' become 'zo' or 'zv'?
            # Even if it's 'zv', we will proceed because the model will learn from audio.
            
            manifest.append({
                "audio_filepath": f"voicecraft_samples/sample_{i}.wav",
                "text": clean_text,
                "phonemes": phonemes_str
            })
            
            if i % 20 == 0:
                print(f"Processed {i} samples...")
                
        except Exception as e:
            print(f"Error at sample {i}: {e}")

    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in manifest:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print("Final Manifest is ready.")

# Run this
prepare_manifest_final_attempt(dataset, "hebrew_train_manifest.jsonl", num_samples=100)

Running final manifest prep for 100 samples...
Processed 0 samples...
Processed 20 samples...
Processed 40 samples...
Processed 60 samples...
Processed 80 samples...
Final Manifest is ready.


In [ ]:
# Read the first line of the manifest to check for Niqqud symbols

with open("hebrew_train_manifest.jsonl", "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())
    text_sample = first_line['text']
    
    print(f"Sample Text: {text_sample}")
    
    # Check if any character in the text is a Niqqud character (Unicode range 05B0-05C7)
    has_niqqud = any('\u05b0' <= char <= '\u05c7' for char in text_sample)
    print(f"Contains Niqqud symbols: {has_niqqud}")

Sample Text: זו הרכישה הגדולה ביותר בתולדות ebay
Contains Niqqud symbols: False


In [ ]:
# Actually expand the vocabulary and see the new phonemes
def expand_vocabulary(manifest_path, original_phn2num):
    # Create a copy so we don't overwrite the original accidentally
    new_phn2num = original_phn2num.copy()
    
    # Start numbering from the next available index
    next_index = max(original_phn2num.values()) + 1
    
    added_phonemes = []
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            # Split the phonemes string back into a list
            phonemes = entry['phonemes'].split()
            for p in phonemes:
                if p not in new_phn2num:
                    new_phn2num[p] = next_index
                    added_phonemes.append(p)
                    next_index += 1
    
    print(f"--- Vocabulary Expansion Complete ---")
    print(f"Added {len(added_phonemes)} new phonemes.")
    print(f"New phonemes examples: {added_phonemes[:10]}")
    return new_phn2num

# Run it!
# Expand vocabulary and save to pkl
new_phn2num = expand_vocabulary("hebrew_train_manifest.jsonl", phn2num)

with open("new_phn2num.pkl", "wb") as f:
    pickle.dump(new_phn2num, f)

print(f"Vocabulary expanded. Total phonemes now: {len(new_phn2num)}")

--- Vocabulary Expansion Complete ---
Added 12 new phonemes.
New phonemes examples: ['ʁ', 'χ', '(', 'en', ')', 'he', 'e', 'a', 'o', 'əʊ']
Vocabulary expanded. Total phonemes now: 92


In [ ]:
# Check what it shout do ??


# Start the fine-tuning process using the new Hebrew manifest
python trainer.py \
    --exp_name hebrew_voicecraft_v1 \
    --config_path config/config.yaml \
    --train_manifest hebrew_train_manifest.jsonl \
    --phn2num_path new_phn2num.pkl

SyntaxError: invalid syntax (17312471.py, line 2)

In [21]:
# Export our new phoneme map to the format VoiceCraft expects (vocab.txt)
with open("vocab.txt", "w", encoding="utf-8") as f:
    # Sort by index to keep it organized
    sorted_vocab = sorted(new_phn2num.items(), key=lambda item: item[1])
    for phn, idx in sorted_vocab:
        f.write(f"{phn} {idx}\n")
print("Generated vocab.txt with 92 phonemes.")

Generated vocab.txt with 92 phonemes.


In [24]:
# Export audio samples from the dataset object to WAV files
output_dir = "voicecraft_samples"
os.makedirs(output_dir, exist_ok=True)

print(f"Saving audio files to {output_dir}...")

for i in range(100):
    try:
        # Get audio array and sampling rate from FLEURS
        audio_array = dataset[i]['audio']['array']
        original_sr = dataset[i]['audio']['sampling_rate']
        
        # Convert to torch tensor
        audio_tensor = torch.from_numpy(audio_array).float().unsqueeze(0)
        
        # Resample to 16000Hz if necessary (VoiceCraft standard)
        if original_sr != 16000:
            resampler = torchaudio.transforms.Resample(original_sr, 16000)
            audio_tensor = resampler(audio_tensor)
        
        # Save file
        file_path = os.path.join(output_dir, f"sample_{i}.wav")
        torchaudio.save(file_path, audio_tensor, 16000)
        
        if i % 20 == 0:
            print(f"Saved {i}/100 samples.")
            
    except Exception as e:
        print(f"Error saving sample {i}: {e}")

print("✅ Done! All WAV files are ready in 'voicecraft_samples'.")

Saving audio files to voicecraft_samples...
Saved 0/100 samples.
Saved 20/100 samples.
Saved 40/100 samples.
Saved 60/100 samples.
Saved 80/100 samples.
✅ Done! All WAV files are ready in 'voicecraft_samples'.


In [25]:
# Save each phoneme sequence to a separate file
phn_output_dir = "voicecraft_data/phonemes"
os.makedirs(phn_output_dir, exist_ok=True)

with open("hebrew_train_manifest.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        with open(os.path.join(phn_output_dir, f"sample_{i}.txt"), "w") as phn_f:
            phn_f.write(data['phonemes'])

print("Phoneme files prepared.")

Phoneme files prepared.


In [31]:
# Create the manifest directory and link our file to 'train.txt'
# !mkdir -p ../voicecraft_data/manifest
!cp hebrew_train_manifest.jsonl ./voicecraft_data/train.txt
# We also need a validation file, for now we can use the same one just to start
!cp hebrew_train_manifest.jsonl ./voicecraft_data/validation.txt

In [33]:
# Convert JSONL manifest to the specific space-separated format VoiceCraft expects

def convert_to_voicecraft_format(input_jsonl, output_txt):
    with open(input_jsonl, 'r', encoding='utf-8') as f_in, \
         open(output_txt, 'w', encoding='utf-8') as f_out:
        for i, line in enumerate(f_in):
            data = json.loads(line)
            # Format: audio_id | speaker_id | phonemes | duration_frames
            # Since we use samples, we'll use sample_{i} as ID
            # For duration, we'll estimate 300 frames (roughly 6 seconds) as a placeholder
            audio_id = f"sample_{i}"
            f_out.write(f"{audio_id} unknown {data['phonemes']} 300\n")

# Ensure the directory exists
os.makedirs("./voicecraft_data/manifest", exist_ok=True)

# Convert our manifest
convert_to_voicecraft_format("./hebrew_train_manifest.jsonl", "./voicecraft_data/manifest/train.txt")
convert_to_voicecraft_format("./hebrew_train_manifest.jsonl", "./voicecraft_data/manifest/validation.txt")

print("Manifests converted to VoiceCraft format.")

Manifests converted to VoiceCraft format.


In [34]:
# Create vocab.txt from our phoneme map
vocab_path = "./voicecraft_data/vocab.txt"
with open(vocab_path, "w", encoding="utf-8") as f:
    # VoiceCraft expects: phoneme index
    for phn, idx in sorted(new_phn2num.items(), key=lambda x: x[1]):
        f.write(f"{phn} {idx}\n")
print(f"Created vocab.txt at {vocab_path}")

Created vocab.txt at ./voicecraft_data/vocab.txt


In [35]:
# Re-convert manifest using Tabs to avoid space-conflicts in phonemes

def convert_to_voicecraft_safe_format(input_jsonl, output_txt):
    with open(input_jsonl, 'r', encoding='utf-8') as f_in, \
         open(output_txt, 'w', encoding='utf-8') as f_out:
        for i, line in enumerate(f_in):
            data = json.loads(line)
            audio_id = f"sample_{i}"
            # Using Tabs (\t) ensures the code reads the duration correctly at the end
            f_out.write(f"{audio_id}\tunknown\t{data['phonemes']}\t300\n")

# Run the conversion again
convert_to_voicecraft_safe_format("./hebrew_train_manifest.jsonl", "./voicecraft_data/manifest/train.txt")
convert_to_voicecraft_safe_format("./hebrew_train_manifest.jsonl", "./voicecraft_data/manifest/validation.txt")

print("Fixed manifest with Tabs created.")

Fixed manifest with Tabs created.


In [36]:
# Re-create vocab.txt with correct order: Index first, then Phoneme
vocab_path = "../voicecraft_data/vocab.txt"

# sorted by index to be safe
sorted_vocab = sorted(new_phn2num.items(), key=lambda x: x[1])

with open(vocab_path, "w", encoding="utf-8") as f:
    for phn, idx in sorted_vocab:
        # Format: index <space> phoneme
        f.write(f"{idx} {phn}\n")

print(f"Fixed vocab.txt at {vocab_path} (Index first, Phoneme second)")

Fixed vocab.txt at ../voicecraft_data/vocab.txt (Index first, Phoneme second)


In [ ]:
# Re-create vocab.txt with a guaranteed format
vocab_path = "../voicecraft_data/vocab.txt"

# sorted by index
sorted_vocab = sorted(new_phn2num.items(), key=lambda x: x[1])

with open(vocab_path, "w", encoding="utf-8") as f:
    for phn, idx in sorted_vocab:
        # We strip any hidden whitespace from the phoneme
        clean_phn = phn.strip()
        # VoiceCraft usually expects: INDEX <space> PHONEME
        f.write(f"{idx} {clean_phn}\n")

print("Vocab file re-written with 'ID PHONEME' format.")
# Let's double check the first line
with open(vocab_path, "r") as f:
    print(f"First line check: {f.readline().strip()}")

Vocab file re-written with 'ID PHONEME' format.
First line check: 0 ɑː


In [38]:
# English comments: Re-writing vocab.txt - strict format: ID then space then Phoneme
vocab_path = "../voicecraft_data/vocab.txt"

with open(vocab_path, "w", encoding="utf-8") as f:
    for phn, idx in sorted(new_phn2num.items(), key=lambda x: x[1]):
        # index comes first, then a single space, then the phoneme
        # making sure there are no extra spaces or tabs
        f.write(f"{int(idx)} {phn.strip()}\n")

print("Vocab fixed: [ID] [SPACE] [PHONEME]")
with open(vocab_path, "r") as f:
    line = f.readline()
    print(f"Final check of first line: '{line.strip()}'")
    # It should look exactly like: '0 ɑː'

Vocab fixed: [ID] [SPACE] [PHONEME]
Final check of first line: '0 ɑː'


In [39]:
# English comments: Creating a completely clean vocab.txt
vocab_path = "../voicecraft_data/vocab.txt"

with open(vocab_path, "w", encoding="utf-8", newline='\n') as f:
    for phn, idx in sorted(new_phn2num.items(), key=lambda x: x[1]):
        # index <space> phoneme
        # Use simple space and ensure no extra invisible characters
        f.write(f"{idx} {phn.strip()}\n")

print("Vocab file written: Index Space Phoneme (Strict format)")

Vocab file written: Index Space Phoneme (Strict format)


In [40]:
# English comments: Re-creating vocab.txt with Phoneme FIRST, ID SECOND
vocab_path = "../voicecraft_data/vocab.txt"

with open(vocab_path, "w", encoding="utf-8", newline='\n') as f:
    for phn, idx in sorted(new_phn2num.items(), key=lambda x: x[1]):
        # [PHONEME] [SPACE] [ID]
        f.write(f"{phn.strip()} {int(idx)}\n")

print("Vocab flipped: Phoneme first, ID second.")
with open(vocab_path, "r") as f:
    print(f"Current first line: '{f.readline().strip()}'") 
    # Should look like: 'ɑː 0'

Vocab flipped: Phoneme first, ID second.
Current first line: 'ɑː 0'


In [41]:
# English comments: Re-writing vocab.txt based on DEBUG insights. 
# The code wants item[0] to be the integer.
vocab_path = "./voicecraft_data/vocab.txt"

with open(vocab_path, "w", encoding="utf-8", newline='\n') as f:
    for phn, idx in sorted(new_phn2num.items(), key=lambda x: x[1]):
        # Format: INDEX <space> PHONEME
        # IMPORTANT: No leading spaces!
        f.write(f"{int(idx)} {phn.strip()}\n")

print("Vocab file fixed based on DEBUG.")

Vocab file fixed based on DEBUG.


In [22]:
cat ./z_scripts/e830M.sh

#!/bin/bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate voicecraft
export CUDA_VISIBLE_DEVICES=0,1,2,3
export WORLD_SIZE=4

dataset=gigaspeech
mkdir -p ./logs/${dataset}

exp_root="path/to/store/exp_results"
exp_name=e830M
dataset_dir="path/to/stored_extracted_codes_and_phonemes/xl" # xs if you only extracted xs in previous step
encodec_codes_folder_name="encodec_16khz_4codebooks"

# export CUDA_LAUNCH_BLOCKING=1 # for debugging

torchrun --nnodes=1 --rdzv-backend=c10d --rdzv-endpoint=localhost:41977 --nproc_per_node=${WORLD_SIZE} \
../main.py \
--reduced_eog 1 \
--drop_long 1 \
--eos 2051 \
--n_special 4 \
--pad_x 0 \
--codebook_weight "[5,1,0.5,0.1]" \
--encodec_sr 50 \
--num_steps 50000 \
--lr 0.05 \
--warmup_fraction 0.01 \
--optimizer_name "ScaledAdam" \
--pseudo_epoch_size 3000 \
--reduce_lr_start_step 3000 \
--reduce_lr_start_epoch 4 \
--clipping_update_period 1000 \
--d_model 2048 \
--audio_embedding_dim 2048 \
--nhead 16 \
--num_decoder_layers 16 \
--max_num_toke

In [ ]:
# Conceptual script to generate Encodec codes if missing
# Load the model (VoiceCraft uses 16khz)
model = EncodecModel.encodec_model_24khz() # or 48khz depending on version
model.set_target_bandwidth(6.0)

In [22]:
# --- PART 0: Path Setup ---
# Since you are in the project root, we use ./ instead of ../
BASE_DATA_DIR = "./voicecraft_data"
CODES_DIR = os.path.join(BASE_DATA_DIR, "encodec_16khz_4codebooks")
MANIFEST_OUT = "hebrew_train_manifest.jsonl"

# --- PART 1: Load Encodec Model ---
print("Loading Encodec model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)
model.to(device)
model.eval()

def finalize_text_for_phonemizer(text):
    # English comments: Normalize Hebrew text to help the phonemizer
    h = Hebrew(text)
    normalized = h.normalize().string
    return normalized

def prepare_everything_final(dataset, output_jsonl, codes_dir, num_samples=100):
    os.makedirs(codes_dir, exist_ok=True)
    manifest = []
    
    print(f"Starting processing for {num_samples} samples from {os.getcwd()}...")
    
    for i in range(num_samples):
        try:
            # --- PART A: Text & Phonemes ---
            raw_text = dataset[i]['transcription']
            clean_text = finalize_text_for_phonemizer(raw_text) # Uses your 'hebrew' library function
            phonemes_list = text_tokenizer_he(clean_text)[0]
            phonemes_str = " ".join(phonemes_list)
            
            # --- PART B: Audio Encoding (.codes) ---
            wav = dataset[i]['audio']['array']
            sr = dataset[i]['audio']['sampling_rate']
            
            # Convert to torch tensor
            wav_tensor = torch.from_numpy(wav).float().unsqueeze(0) 
            if wav_tensor.shape[0] > 1: # ensure mono
                wav_tensor = wav_tensor.mean(dim=0, keepdim=True)
            
            # Resample to 24khz for Encodec
            wav_24k = convert_audio(wav_tensor, sr, 24000, 1)
            wav_24k = wav_24k.to(device)
            
            with torch.no_grad():
                encoded_frames = model.encode(wav_24k.unsqueeze(0))
                codes = torch.cat([frame[0] for frame in encoded_frames], dim=-1).squeeze(0) 
            
            # Save codes to disk
            code_filename = f"sample_{i}.codes"
            torch.save(codes.cpu(), os.path.join(codes_dir, code_filename))
            
            # --- PART C: Manifest entry ---
            manifest.append({
                "audio_filepath": f"sample_{i}", 
                "text": clean_text,
                "phonemes": phonemes_str,
                "duration": len(wav) / sr
            })
            
            if i % 10 == 0:
                print(f"✅ Processed {i}/{num_samples} samples...")
                
        except Exception as e:
            print(f"❌ Error at sample {i}: {e}")

    # Save manifest
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for entry in manifest:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            
    print(f"\n✨ Done! Files are in {BASE_DATA_DIR}")

# Running the process
prepare_everything_final(dataset, MANIFEST_OUT, CODES_DIR, num_samples=100)

Loading Encodec model...
Starting processing for 100 samples from /home/sukiennik/projects/VoiceCraft...


/home/sukiennik/miniconda3/envs/voicecraft_linux/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv1d(input, weight, bias, self.stride,


✅ Processed 0/100 samples...
✅ Processed 10/100 samples...
✅ Processed 20/100 samples...
✅ Processed 30/100 samples...
✅ Processed 40/100 samples...
✅ Processed 50/100 samples...
✅ Processed 60/100 samples...
✅ Processed 70/100 samples...
✅ Processed 80/100 samples...
✅ Processed 90/100 samples...

✨ Done! Files are in ./voicecraft_data


In [23]:
# English comments: Generate a vocab.txt file based on the actual phonemes in the manifest
phonemes = [
    '!', '"', '(', ')', ',', '.', ':', '_', 'a', 'aɪ', 'b', 'd', 'dʒ', 'e', 'en', 'eɪ', 
    'f', 'h', 'he', 'i', 'iː', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'tʃ', 'u', 
    'uː', 'v', 'w', 'z', 'ŋ', 'ɑː', 'əʊ', 'ɛ', 'ɡ', 'ɹ', 'ʁ', 'ʃ', 'ʔ', 'χ'
]

vocab_path = "./voicecraft_data/vocab.txt"

with open(vocab_path, "w", encoding="utf-8") as f:
    for i, phn in enumerate(phonemes):
        f.write(f"{i} {phn}\n")
    # Add a special token for padding at the end
    f.write(f"{len(phonemes)} <pad>\n")

print(f"✅ Vocab created with {len(phonemes) + 1} tokens.")

✅ Vocab created with 47 tokens.


In [27]:
# English comments: Re-formatting the manifest so the second column matches the ID
import os

base_dir = "./voicecraft_data"
old_manifest = os.path.join(base_dir, "manifest/train.txt")
new_manifest = os.path.join(base_dir, "manifest/train_fixed.txt")
phonemes_dir = os.path.join(base_dir, "phonemes")
os.makedirs(phonemes_dir, exist_ok=True)

with open(old_manifest, "r", encoding="utf-8") as f_in, \
     open(new_manifest, "w", encoding="utf-8") as f_out:
    
    for line in f_in:
        parts = line.strip().split("\t")
        if len(parts) >= 3:
            file_id = parts[0]
            phonemes = parts[2]
            duration = parts[3] if len(parts) > 3 else "10.0"
            
            # 1. Update manifest: ID <TAB> ID <TAB> PHONEMES <TAB> DURATION
            # This ensures the 'speaker' column is also the unique ID
            f_out.write(f"{file_id}\t{file_id}\t{phonemes}\t{duration}\n")
            
            # 2. Create the phoneme file (sample_0.txt)
            with open(os.path.join(phonemes_dir, f"{file_id}.txt"), "w", encoding="utf-8") as phn_f:
                phn_f.write(phonemes)

# Replace the old manifest with the fixed one
os.replace(new_manifest, old_manifest)
# Create a validation copy
import shutil
shutil.copy(old_manifest, os.path.join(base_dir, "manifest/validation.txt"))

print("✅ Manifest fixed: Speaker column now matches ID column.")
print(f"✅ Phoneme files created in {phonemes_dir}")

✅ Manifest fixed: Speaker column now matches ID column.
✅ Phoneme files created in ./voicecraft_data/phonemes


In [ ]:
# English comments: Converting binary .txt (formerly .codes) files to plain text format


encodec_dir = "./voicecraft_data/encodec_16khz_4codebooks"
files = [f for f in os.listdir(encodec_dir) if f.endswith(".txt")]
converted_count = 0

print("Converting binary files to plain text...")

for filename in files:
    file_path = os.path.join(encodec_dir, filename)
    try:
        # Try to load as a torch tensor/pickle
        data = torch.load(file_path)
        
        # If it's a tensor, convert to list or numpy
        if torch.is_tensor(data):
            data = data.cpu().numpy()
        
        # VoiceCraft expects codes to be space-separated or comma-separated integers
        # We will format it as space-separated rows (one row per codebook)
        if hasattr(data, 'tolist'):
            lines = []
            for row in data:
                lines.append(" ".join(map(str, row)))
            text_content = "\n".join(lines)
            
            # Save back as real plain text
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text_content)
            converted_count += 1
            
    except Exception as e:
        # If it's already text, it might fail to torch.load, which is fine
        pass

print(f"✅ Successfully converted {converted_count} files to plain text.")

# Now let's try to read the first file again to verify
sample_path = os.path.join(encodec_dir, "sample_0.txt")
with open(sample_path, 'r') as f:
    print(f"📄 New sample content: {f.read()[:50]}...")

Converting binary files to plain text...
✅ Successfully converted 100 files to plain text.
📄 New sample content: 62 62 62 62 408 408 408 408 62 408 408 408 408 835...


In [32]:
torch.cuda.empty_cache()

In [35]:
# English comments: Manually setting up arguments and starting training within the notebook
import sys
import argparse
from main import get_args # Assuming main.py has a get_args function, or we'll mock it
import torch
from steps.trainer import Trainer

# 1. Manually create the Namespace object with all required parameters
args = argparse.Namespace(
    seed=1,
    precision='float16',
    num_workers=4,
    resume=False,
    tb_write_every_n_steps=100,
    print_every_n_steps=10,
    val_every_n_steps=500,
    lr=0.0005,
    batch_size=4,
    max_num_tokens=100000,
    val_max_num_tokens=None,
    num_buckets=6,
    dynamic_batching=0,
    weight_decay=0.01,
    warmup_fraction=0.01,
    num_epochs=10,
    num_steps=5000,
    gradient_accumulation_steps=4,
    gradient_clip_val=1.0,
    early_stop_step=3200,
    early_stop_threshold=-1.0,
    optimizer_name='AdamW',
    reduce_lr_start_step=3000,
    pseudo_epoch_size=3000,
    reduce_lr_start_epoch=4,
    clipping_update_period=600,
    exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M',
    dataset='gigaspeech',
    dataset_dir='./voicecraft_data',
    phn_folder_name='phonemes',
    encodec_folder_name='encodec_16khz_4codebooks',
    manifest_name='manifest',
    pad_x=1,
    audio_max_length=20,
    audio_min_length=2,
    text_max_length=400,
    text_min_length=10,
    encodec_sr=50,
    drop_long=0,
    mask_len_min=1,
    mask_len_max=600,
    eos=-1,
    reduced_eog=0,
    special_first=0,
    n_special=3,
    codebook_weight=None,
    max_mask_portion=0.7,
    max_n_spans=3,
    shuffle_mask_embedding=0,
    mask_sample_dist='poisson1',
    min_gap=5,
    n_codebooks=4,
    text_vocab_size=46,
    text_pad_token=46,
    audio_vocab_size=2048,
    empty_token=2048,
    eog=2049,
    audio_pad_token=2050,
    d_model=2048,
    audio_embedding_dim=2048,
    text_embedding_dropout=0.1,
    audio_embedding_dropout=0,
    text_positional_embedding_dropout=0.1,
    audio_positional_embedding_dropout=0.1,
    trm_dropout=0.1,
    nhead=16,
    num_decoder_layers=16,
    load_model_from=None,
    sep_special_token=0 # <--- INJECTED MANUALLY HERE
)

# 2. Initialize and Start Training
print("🚀 Starting Trainer manually from notebook...")
my_trainer = Trainer(args)
my_trainer.train()

ImportError: cannot import name 'get_args' from 'main' (/home/sukiennik/projects/VoiceCraft/main.py)

In [ ]:
!torchrun --nproc_per_node=1 main.py \
--dataset_dir "./voicecraft_data" \
--exp_dir "./experiments/hebrew_fleurs/hebrew_v1_330M" \
--dataset "gigaspeech" \
--manifest_name "manifest" \
--text_vocab_size 46 \
--text_pad_token 46 \
--num_steps 5000 \
--lr 0.0005 \
--batch_size 4 \
--gradient_accumulation_steps 4 \
--n_codebooks 4 \
--audio_vocab_size 2048 \
--num_workers 4 \
--val_every_n_steps 500 \
--print_every_n_steps 10

2026-02-23 09:51:10,089 [INFO] main.py:19 || Namespace(seed=1, precision='float16', num_workers=4, resume=False, tb_write_every_n_steps=100, print_every_n_steps=10, val_every_n_steps=500, lr=0.0005, batch_size=4, max_num_tokens=100000, val_max_num_tokens=None, num_buckets=6, dynamic_batching=0, weight_decay=0.01, warmup_fraction=0.01, num_epochs=10, num_steps=5000, gradient_accumulation_steps=4, gradient_clip_val=1.0, early_stop_step=3200, early_stop_threshold=-1.0, optimizer_name='AdamW', reduce_lr_start_step=3000, pseudo_epoch_size=3000, reduce_lr_start_epoch=4, clipping_update_period=600, exp_dir='./experiments/hebrew_fleurs/hebrew_v1_330M', dataset='gigaspeech', dataset_dir='./voicecraft_data', phn_folder_name='phonemes', encodec_folder_name='encodec_16khz_4codebooks', manifest_name='manifest', pad_x=1, audio_max_length=20, audio_min_length=2, text_max_length=400, text_min_length=10, encodec_sr=50, drop_long=0, mask_len_min=1, mask_len_max=600, eos=-1, reduced_eog=0, special_first=

In [ ]:
# Replace with the actual model path you are using in VoiceCraft
model_id = "pydub/voicecraft" # example path
tokenizer = AutoTokenizer.from_pretrained(model_id)

test_text = "שלום, אני בודקת עברית"
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)

print(f"Original: {test_text}")
print(f"Decoded:  {decoded}")
print(f"Tokens:   {tokens}")

if "[UNK]" in decoded or "" in decoded:
    print("\n⚠️ Warning: Tokenizer does not support Hebrew!")